# Lab Exploration 3 - G-Eval

For this lab, you will experiment with G-Eval. G-Eval is an LLM-as-a-judge framework that uses chain-of-thought prompting under the hood to evaluate on any custom criteria. You should be familiar with the respective paper we are reading in class: [G-EVAL: NLG Evaluation using GPT-4 with Better Human Alignment](https://arxiv.org/pdf/2303.16634)



G-Eval is available through DeepEval, an evaluation platform. You should start by spending some time reading through the G-Eval docs to get an understanding of how the library works: [G-Eval](https://deepeval.com/docs/metrics-llm-evals).

We are expecting you to spend ~5 hours on this assignment exploring and trying to use G-Eval.

For this lab, your tasks are to:
- Read through the G-Eval documentation: https://deepeval.com/docs/metrics-llm-evals
- Get the sample code below working
- Pick at least 3 metrics (correctness, coherence, tonality, creativity, etc., whatever you can think of) that you think would be interesting to experiment with.
- Pick at least 1 model to experiment with. The example below uses Qwen.
- Setup your metrics and experiment with them. You should explore the following questions:
  1. What is the difference between providing criteria and evaluation_steps? Do you notice performance differences?
  2. How does providing a rubric affect evaluations?
  3. How well does the model perform on each metric? Do you agree with the model's assessments? Would you trust this LLM-as-a-judge approach at scale?
  4. If you are dissatisfied with the performance of the judge, what can you do to make it better? Is there something you need to change in the criteria/evaluation_steps, test_case, or something else?

Some other options for exploration:
- Compare several different LLM judges. You can find many different kinds of models on HuggingFace. How do they differ in performance and judgement?
- Find or create a dataset of 5-10 samples for a given metric, run G-Eval across the data, and also complete the task yourself. What is your inter-rater agreement with the LLM?
- [GEval has an option to override the default templates](https://deepeval.com/docs/metrics-llm-evals#customize-your-template). This may be interesting to experiment with.

Below is some sample code for getting starting using G-Eval:

In [ ]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.3/819.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemetry-proto-1.38.0:
      Successfully uninstalled opentelemetry-proto-1.38.0


In [ ]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from deepeval.models.base_model import DeepEvalBaseLLM

By default DeepEval looks for an OpenAI API key to use as the backend LLM in G-Eval. However, you can also load HuggingFace models:

In [ ]:
class HFModel(DeepEvalBaseLLM):
    def __init__(self, model_name):
        self.device = "cuda" # the device to load the model onto
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model_name = model_name

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        model = self.load_model()

        messages = [
            {"role": "user", "content": prompt}
        ]

        model_inputs = self.tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True
        ).to(self.device)

        generated_ids = model.generate(input_ids=model_inputs['input_ids'], attention_mask=model_inputs['attention_mask'], max_new_tokens=300, do_sample=True)
        # Only decode the new tokens, not the input prompt
        response_ids = generated_ids[:, model_inputs['input_ids'].shape[1]:]
        return self.tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return self.model_name

Here are a few models you can start out experimenting with. You should be able to load all of these on a 40GB A100. I was also able to run all of these on a 15GB T4, but Qwen is pretty tight.

Heads up: you will likely find that TinyLlama does not work well with G-Eval, because it's not very good at returning properly formatted JSON. [There is apparently a way that you can improve this](https://deepeval.com/guides/guides-using-custom-llms#json-confinement-libraries), but I couldn't get it to work. If you don't have enough resources to run Qwen, you can 1). Try to find a smaller model in the 1B-3B parameter range that works, 2). try to use a JSON confinement library, or 3). ditch G-Eval and experiment with LLM-as-a-judge through pure prompting. This is also a valid approach for this assignment.

In [ ]:
tinyLlama = HFModel("TinyLlama/TinyLlama-1.1B-Chat-v1.0") # 1.1B parameters

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
qwen = HFModel("Qwen/Qwen2.5-7B-Instruct") # 7B parameters

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# We can just test calling the models directly
testText = "Evaluate the creativity of this sentence on a scale of 1-10: 'Now the other princes of the Achaeans slept soundly the whole night through, but Agamemnon son of Atreus was troubled, so that he could get no rest.'"

print("TinyLlama:")
print(tinyLlama.generate(testText))

print("Qwen:")
print(qwen.generate(testText))

TinyLlama:
I can't judge the creativity of a given sentence, but I can answer it based on what the sentence contains. "Now the other princes of the Achaeans slept soundly the whole night through, but Agamemnon son of Atreus was troubled, so that he could get no rest." The sentence incorporates multiple elements such as action, time, state of being, mood, and expression and expresses a range of meanings. The sentence is not too complex, yet it does convey various meanings and creates a vivid picture of the scene. Therefore, I give the sentence a 7 out of 10 points on a scale of 1-10.
Qwen:
To evaluate the creativity of the sentence, let's break it down:

1. **Imagery and Detail**: The sentence paints a vivid picture by contrasting the peaceful sleep of the other princes with Agamemnon's unrest. This use of imagery is quite effective.

2. **Structure and Flow**: The sentence flows well, using parallel structure ("Now the other princes... but Agamemnon...") which adds to its rhythmic qual

Now, let's try GEval.

Notice that GEval can take either a `criteria` parameter or `evaluation_steps`. If it's only given `criteria`, it will generate it's own `evaluation_steps`, like what is done in the original paper.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

name = "Accuracy"
criteria = "Determine whether the actual output is factually accurate and correct based on the expected output. Vague language or syntactic difference are OK. Ignore sentence structure differences."

correctness_metric = GEval(
    name=name,
    criteria=criteria,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model=qwen,
    verbose_mode=True
)

In [ ]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input="The dog chased the cat up the tree, who ran up the tree?",
    actual_output="It would be the cat.",
    expected_output="The cat."
)

In [ ]:
correctness_metric.measure(test_case)

Output()

**************************************************

Accuracy [GEval] Verbose Logs

**************************************************

Criteria:
Determine whether the actual output is factually accurate and correct based on the expected output. Vague language 
or syntactic difference are OK. Ignore sentence structure differences. 
 
Evaluation Steps:
[
    "Check if the actual output contains information that directly contradicts the expected output.",
    "Verify if the core facts and details in the actual output match those in the expected output, ignoring 
differences in sentence structure or phrasing.",
    "Assess whether the actual output covers all the key points mentioned in the expected output."
] 
 
Rubric:
None 
 
Score: 0.5
 
Reason: The actual output 'It would be the cat.' does not contradict the expected output 'The cat.', but it does 
not fully cover all key points as it adds unnecessary phrasing. The core fact matches, but the evaluation criteria 
for covering all key points is not met.

======================================================================

0.5

In [ ]:
# Let's try a different metric
name = "Creativity"
criteria = "Determine how creative the given sentence is, on a scale of 1-10."

creativity_metric = GEval(
    name=name,
    criteria=criteria,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=qwen,
    verbose_mode=True
)

In [ ]:
test_case_2 = LLMTestCase(
    input="", # Add an empty string for the required input parameter
    actual_output="Now the other princes of the Achaeans slept soundly the whole night through, but Agamemnon son of Atreus was troubled, so that he could get no rest."
)

In [ ]:
creativity_metric.measure(test_case_2)

Output()

**************************************************

Creativity [GEval] Verbose Logs

**************************************************

Criteria:
Determine how creative the given sentence is, on a scale of 1-10. 
 
Evaluation Steps:
[
    "Count the number of unique ideas or perspectives expressed in the sentence.",
    "Assess the originality and uniqueness of the content compared to typical expressions.",
    "Evaluate the use of metaphors, similes, or other literary devices that enhance creativity.",
    "Consider the overall impact and memorability of the sentence."
] 
 
Rubric:
None 
 
Score: 0.2
 
Reason: The response does not express multiple unique ideas or perspectives, lacks originality, does not use 
literary devices, and has a straightforward narrative style without enhancing memorability.

======================================================================

0.2

In [ ]:
# Experimentation here